# Llama 2
"Llama 2: Open Foundation and Fine-Tuned Chat Models" (Touvron et al., 2023)

**Idea:** the decoder recipe that became the open-model default: pre-norm with RMSNorm, SwiGLU, RoPE. `slm.ipynb` already builds that block. This notebook adds the three things Llama 2 needs to be *served* efficiently, which `slm.ipynb` skipped:
1. **Grouped-query attention (GQA)**: several query heads share one key/value head.
2. **KV cache**: generate each new token without recomputing the past.
3. **Top-p (nucleus) sampling**: the sampler used by the reference code.

### Llama 2 vs the SLM notebook
| | `slm.ipynb` | Llama 2 (this notebook) |
|---|---|---|
| Norm / FFN / position | RMSNorm / SwiGLU / RoPE | same |
| Attention heads | $h$ query heads, $h$ key/value heads (MHA) | $h$ query heads, $h_{kv} < h$ key/value heads (**GQA**; in the 34B and 70B models) |
| Generation | re-runs the whole context for every token: $O(T^2)$ per token | **KV cache**: only the new token is processed, $O(T)$ per token |
| Sampling | temperature | temperature + **top-p** |
| Sizes | 5M params | 7B / 13B / 70B, 4k context, 2T training tokens |

Shape flow: `idx (B, T) -> embed (B, T, D) -> [RMSNorm -> GQA(RoPE, cache) -> + -> RMSNorm -> SwiGLU -> +] x L -> RMSNorm -> logits (B, T, V)`

In [ ]:
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(42)

## 1. Data and shared pieces (same as `slm.ipynb`)
Character-level tiny Shakespeare with the same split and batch shape, so val losses are comparable with `slm.ipynb`. RMSNorm, RoPE and SwiGLU are copied unchanged; see `slm.ipynb` and `rope.ipynb` for their formulas.

In [ ]:
text = open("../datasets/tiny_shakespeare.txt").read()
chars = sorted(set(text))
vocab_size = len(chars)
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for c, i in stoi.items()}
encode = lambda s: [stoi[c] for c in s]
decode = lambda ids: "".join(itos[i] for i in ids)

data = torch.tensor(encode(text))
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]
block_size, batch_size = 128, 32   # T, B

def get_batch(split):
    d = train_data if split == "train" else val_data
    ix = torch.randint(len(d) - block_size - 1, (batch_size,))
    x = torch.stack([d[i:i + block_size] for i in ix])
    y = torch.stack([d[i + 1:i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)


class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps) * self.weight


def rope_cache(T, head_dim, base=10000):
    freqs = 1.0 / base ** (torch.arange(0, head_dim, 2).float() / head_dim)
    angles = torch.arange(T).float()[:, None] * freqs[None]                  # (T, head_dim/2)
    return angles.cos(), angles.sin()


def apply_rope(x, cos, sin):
    x1, x2 = x[..., 0::2], x[..., 1::2]
    return torch.stack([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1).flatten(-2)


class SwiGLU(nn.Module):
    def __init__(self, dim, hidden):
        super().__init__()
        self.w1 = nn.Linear(dim, hidden, bias=False)
        self.w3 = nn.Linear(dim, hidden, bias=False)
        self.w2 = nn.Linear(hidden, dim, bias=False)

    def forward(self, x):
        return self.w2(F.silu(self.w1(x)) * self.w3(x))


def ffn_hidden(dim):
    return 64 * ((8 * dim // 3 + 63) // 64)

## 2. Grouped-query attention
With a KV cache (§3), the memory that grows during generation is the keys and values of every past token, in every layer and head. GQA shrinks exactly that: keep all $h$ **query** heads, but compute only $h_{kv}$ **key/value** heads and let each group of $g = h / h_{kv}$ query heads share one.

### Formula: Grouped-query attention
$$\text{head}_i = \text{Attention}\big(X W_Q^{\,i},\ X W_K^{\,\lfloor i / g \rfloor},\ X W_V^{\,\lfloor i / g \rfloor}\big), \qquad i = 0, \dots, h - 1, \quad g = h / h_{kv}$$
- $h = 8$ query heads, $h_{kv} = 2$ key/value heads, so $g = 4$ query heads per group
- $h_{kv} = h$: ordinary multi-head attention (MHA). $h_{kv} = 1$: multi-query attention (MQA)
- $W_K, W_V \in \mathbb{R}^{D \times h_{kv} d}$ instead of $D \times h d$: fewer parameters, and a cache $h / h_{kv}$ times smaller
- Llama 2 70B: $h = 64$, $h_{kv} = 8$. Quality is close to MHA; MQA loses more (Ainslie et al., 2023)

In code the $h_{kv}$ key/value heads are simply repeated $g$ times before the usual attention.

In [ ]:
class GroupedQueryAttention(nn.Module):
    # x (B, T, D) -> (B, T, D); cache = (k, v) of earlier tokens, each (B, kv_heads, T_past, hd)
    def __init__(self, dim, num_heads, num_kv_heads):
        super().__init__()
        assert num_heads % num_kv_heads == 0
        self.num_heads, self.num_kv_heads = num_heads, num_kv_heads
        self.head_dim = dim // num_heads
        self.wq = nn.Linear(dim, num_heads * self.head_dim, bias=False)
        self.wk = nn.Linear(dim, num_kv_heads * self.head_dim, bias=False)   # smaller than wq
        self.wv = nn.Linear(dim, num_kv_heads * self.head_dim, bias=False)
        self.proj = nn.Linear(dim, dim, bias=False)

    def forward(self, x, cos, sin, cache=None):
        B, T, D = x.shape
        q = self.wq(x).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)      # (B, h, T, hd)
        k = self.wk(x).view(B, T, self.num_kv_heads, self.head_dim).transpose(1, 2)   # (B, h_kv, T, hd)
        v = self.wv(x).view(B, T, self.num_kv_heads, self.head_dim).transpose(1, 2)   # (B, h_kv, T, hd)
        q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)                       # cos/sin are for the NEW positions

        if cache is not None:                                    # prepend the stored keys/values of past tokens
            k, v = torch.cat([cache[0], k], dim=2), torch.cat([cache[1], v], dim=2)   # (B, h_kv, T_past + T, hd)
        new_cache = (k, v)                                       # stored with h_kv heads, already rotated

        g = self.num_heads // self.num_kv_heads
        k, v = k.repeat_interleave(g, dim=1), v.repeat_interleave(g, dim=1)           # (B, h, T_total, hd)

        att = (q @ k.transpose(-2, -1)) / self.head_dim ** 0.5                        # (B, h, T, T_total)
        T_total = k.shape[2]
        mask = torch.ones(T_total, T_total, dtype=torch.bool, device=x.device).tril()[-T:]   # (T, T_total): rows of the new tokens
        att = att.masked_fill(~mask, float("-inf")).softmax(dim=-1)
        out = (att @ v).transpose(1, 2).reshape(B, T, D)
        return self.proj(out), new_cache

mha = GroupedQueryAttention(256, 8, 8)
gqa = GroupedQueryAttention(256, 8, 2)
count = lambda m: sum(p.numel() for p in m.parameters())
print(f"attention params | MHA (8 kv heads): {count(mha)} | GQA (2 kv heads): {count(gqa)}")
cos, sin = rope_cache(10, 32)
out, cache = gqa(torch.randn(2, 10, 256), cos, sin)
print("output:", out.shape, "| cached keys:", cache[0].shape, "(2 kv heads instead of 8)")

## 3. KV cache
Without a cache, producing token $t + 1$ runs the model over all $t$ previous tokens again, although their keys and values have not changed: with a causal mask, token $i$'s representation never depends on later tokens.

So: keep each layer's $K$ and $V$ for the tokens seen so far, feed **only the new token**, compute its $q, k, v$, append its $k, v$ to the cache and attend over the whole cache.

### Formula: One cached decoding step
$$K_{\le t} = \big[K_{< t};\ k_t\big], \quad V_{\le t} = \big[V_{< t};\ v_t\big], \qquad o_t = \text{softmax}\!\left(\frac{q_t K_{\le t}^T}{\sqrt{d}}\right) V_{\le t}$$
- $q_t, k_t, v_t$: computed from the new token only, rotated by RoPE for position $t$
- no mask is needed: everything in the cache is in the past
- cost per new token: $O(t)$ instead of $O(t^2)$; memory: $2 \cdot L \cdot h_{kv} \cdot t \cdot d$ floats, which is what GQA reduces
- cached keys were rotated by their own position when stored and are never touched again (a property of RoPE)

The model below takes `start_pos` so the new tokens get the right RoPE angles.

In [ ]:
class Block(nn.Module):
    def __init__(self, dim, num_heads, num_kv_heads):
        super().__init__()
        self.norm1, self.attn = RMSNorm(dim), GroupedQueryAttention(dim, num_heads, num_kv_heads)
        self.norm2, self.ffn = RMSNorm(dim), SwiGLU(dim, ffn_hidden(dim))

    def forward(self, x, cos, sin, cache=None):
        a, cache = self.attn(self.norm1(x), cos, sin, cache)
        x = x + a
        return x + self.ffn(self.norm2(x)), cache


class Llama(nn.Module):
    # idx (B, T) -> logits (B, T, V), caches (one (k, v) per layer)
    def __init__(self, vocab_size, max_len=1024, dim=256, depth=6, num_heads=8, num_kv_heads=2):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, dim)
        self.blocks = nn.ModuleList([Block(dim, num_heads, num_kv_heads) for _ in range(depth)])
        self.norm = RMSNorm(dim)
        self.head = nn.Linear(dim, vocab_size, bias=False)
        self.head.weight = self.embed.weight
        nn.init.normal_(self.embed.weight, std=0.02)
        cos, sin = rope_cache(max_len, dim // num_heads)
        self.register_buffer("cos", cos)
        self.register_buffer("sin", sin)

    def forward(self, idx, caches=None, start_pos=0):
        T = idx.shape[1]
        cos, sin = self.cos[start_pos:start_pos + T], self.sin[start_pos:start_pos + T]   # angles of the new positions
        x = self.embed(idx)
        new_caches = []
        for i, block in enumerate(self.blocks):
            x, c = block(x, cos, sin, None if caches is None else caches[i])
            new_caches.append(c)
        return self.head(self.norm(x)), new_caches

model = Llama(vocab_size).to(device)
print(f"params: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M")

# check: feeding tokens one at a time through the cache gives the same logits as one full pass
model.eval()
x = get_batch("val")[0][:1, :24]                                  # (1, 24)
with torch.no_grad():
    full, _ = model(x)                                            # (1, 24, V)
    caches, steps = None, []
    for t in range(x.shape[1]):
        logits, caches = model(x[:, t:t + 1], caches, start_pos=t)   # (1, 1, V)
        steps.append(logits)
print("max |full pass - cached steps|:", (full - torch.cat(steps, dim=1)).abs().max().item())
model.train()

## 4. Training
Identical to `slm.ipynb` (no cache during training: all positions are computed in one pass with the causal mask). See there for the loss and clipping formulas.

In [ ]:
max_iters = 3000
eval_every = 500
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

def loss_fn(model, x, y):
    logits, _ = model(x)
    return F.cross_entropy(logits.reshape(-1, vocab_size), y.reshape(-1))

@torch.no_grad()
def estimate_loss(model, iters=50):
    model.eval()
    out = {s: sum(loss_fn(model, *get_batch(s)).item() for _ in range(iters)) / iters for s in ["train", "val"]}
    model.train()
    return out

for it in range(max_iters + 1):
    if it % eval_every == 0:
        l = estimate_loss(model)
        print(f"iter {it:5d} | train {l['train']:.4f} | val {l['val']:.4f}")
    if it == max_iters:
        break
    loss = loss_fn(model, *get_batch("train"))
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()

## 5. Top-p (nucleus) sampling
Top-k keeps a fixed number of candidates. But sometimes the model is sure (one sensible next token) and sometimes it is not (hundreds). Top-p keeps the **smallest set of tokens whose probabilities add up to $p$**, so the candidate set shrinks and grows with the model's confidence.

### Formula: Top-p sampling
$$\mathcal{V}_p = \text{smallest set with} \sum_{i \in \mathcal{V}_p} p_i \ge p, \qquad \tilde p_i = \begin{cases} p_i \big/ \sum_{j \in \mathcal{V}_p} p_j & i \in \mathcal{V}_p \\ 0 & \text{otherwise} \end{cases}$$
- $p_i = \text{softmax}(z / \tau)_i$: next-token probabilities after temperature $\tau$
- sort descending, take the cumulative sum, drop every token whose *preceding* mass already exceeds $p$ (so the top token is always kept)
- Llama 2's reference defaults: $\tau = 0.6$, $p = 0.9$

In [ ]:
def sample_top_p(probs, p):
    # probs (B, V) -> sampled ids (B, 1)
    sorted_p, sorted_i = probs.sort(dim=-1, descending=True)        # (B, V)
    mass_before = sorted_p.cumsum(dim=-1) - sorted_p                # mass of the tokens ranked above each one
    sorted_p = sorted_p.masked_fill(mass_before > p, 0.0)           # outside the nucleus
    choice = torch.multinomial(sorted_p / sorted_p.sum(-1, keepdim=True), 1)   # (B, 1) index into the sorted order
    return sorted_i.gather(-1, choice)

probs = torch.tensor([[0.5, 0.3, 0.1, 0.05, 0.05]])
draws = torch.cat([sample_top_p(probs, 0.75) for _ in range(2000)]).flatten()
print("p = 0.75 keeps tokens {0, 1}; sampled shares:", (torch.bincount(draws, minlength=5) / 2000).tolist())

## 6. Generation with and without the cache
`generate` feeds the prompt once to fill the cache, then one token per step. Because position comes from RoPE and the cache, it is not limited to `block_size` tokens (up to `max_len`, the size of the RoPE table).

The timing compares it with the `slm.ipynb` way (re-run the whole context for each new token). Both produce the same distribution; only the cost differs. On a model this small, per-call overhead hides the gain for short outputs; the gap opens as the sequence grows (per-token cost $O(t)$ against $O(t^2)$) and is decisive at real sizes.

In [ ]:
@torch.no_grad()
def generate(model, start="\n", n=500, temperature=0.8, top_p=0.9, use_cache=True):
    model.eval()
    idx = torch.tensor([encode(start)], device=device)                       # (1, t)
    logits, caches = model(idx)                                              # prompt pass fills the cache
    for _ in range(n):
        nxt = sample_top_p(F.softmax(logits[:, -1] / temperature, dim=-1), top_p)   # (1, 1)
        idx = torch.cat([idx, nxt], dim=1)
        if use_cache:
            logits, caches = model(nxt, caches, start_pos=idx.shape[1] - 1)  # only the new token
        else:
            logits, _ = model(idx)                                           # everything again
    model.train()
    return decode(idx[0].tolist())

print(generate(model))

for use_cache in [False, True]:
    torch.manual_seed(0)
    t0 = time.time()
    generate(model, n=900, use_cache=use_cache)
    print(f"use_cache = {use_cache!s:5} | 900 tokens in {time.time() - t0:.2f}s")

cache_floats = lambda kv_heads, T=4096, L=80, hd=128: 2 * L * kv_heads * T * hd
print(f"Llama 2 70B cache at 4096 tokens, fp16: MHA (64 kv heads) {cache_floats(64) * 2 / 2**30:.1f} GB | GQA (8 kv heads) {cache_floats(8) * 2 / 2**30:.1f} GB per sequence")

## Notes
- Architecturally Llama 2 is Llama 1 with a longer context (4k) and GQA in the large models; the gains came from 40% more training data and from the chat fine-tuning. Sizes: 7B ($L = 32$, $D = 4096$, 32 heads), 13B, 70B ($L = 80$, $D = 8192$, 64 heads, 8 KV heads).
- **Llama 2-Chat** is the base model after supervised fine-tuning on dialogues and then RLHF with two reward models (helpfulness and safety), using the prompt format `[INST] <<SYS>> system prompt <</SYS>> user message [/INST] answer`. See the post-training section of `models.md`.
- The KV cache is why serving cost is dominated by memory, and why prompt caching, GQA, sliding windows (`mistral.ipynb`) and paged attention (vLLM) exist. With batching, the cache is per sequence.
- The cache also splits generation into two phases with different costs: **prefill** (the prompt, processed in parallel) and **decode** (one token at a time).
- Tokenizer: SentencePiece BPE with a 32k vocabulary. Character-level here, as in `slm.ipynb`.